# LegalLens on Colab GPU — improved pipeline (A–D)

Runs the LegalLens summarisation pipeline with the four outcome-preservation fixes:

- **A.** Disposition detection now recognises civil money decrees (`held liable`, `ordered to refund`, `the suit is decreed`, `compensation`/`damages`), not only appeal/criminal forms.
- **B.** The formatter reserves the **judgement** section's words first, so the final order is never trimmed out when a document is over budget.
- **C.** Any operative-order paragraph in the document tail is promoted to a dedicated `judgement` slot — even when section detection falls back to a single block.
- **D.** Evaluation reports two new signals per document: `outcome_present` (did the summary state the order at all?) and `outcome_recall` (token recall of the reference's disposition).

> **Runtime:** set **Runtime → Change runtime type → GPU (T4)** before running.

> **Note on checkpoints:** the fine-tuned BART checkpoints (`models/legallens-bart-base-e*`) are git-ignored, so a fresh `git clone` will **not** include them. Either (1) upload a checkpoint to `models/`, or (2) train one with the optional training section below. If no checkpoint is present, the evaluation cell automatically runs the zero-shot baseline alone — fixes A–D still apply, since they are inference-time, model-independent changes.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Get the code

**Option A — clone (if you have a git remote).** Fill in your repo URL and branch.

**Option B — upload a zip.** Zip the `legal-doc-intelligence` folder locally, use the Colab file browser (📁 on the left) to upload it, then run:
```python
!unzip -q legal-doc-intelligence.zip
```

**Option C — Google Drive.** `from google.colab import drive; drive.mount('/content/drive')` then `%cd` into the project copy on your Drive.

In [ ]:
import os

# --- Option A: clone ---
REPO_URL = "https://github.com/Phyboc/Legal-Document-Summarizer.git"  # set to "" to use Option B/C
BRANCH   = "review2"   # must be the branch that has the A–D changes + this notebook (commit & push first!)
# Private repo? Clone with a token instead:
#   REPO_URL = "https://<GITHUB_TOKEN>@github.com/Phyboc/Legal-Document-Summarizer.git"

if REPO_URL:
    !rm -rf legallens_repo
    !git clone --branch {BRANCH} {REPO_URL} legallens_repo
    # legal-doc-intelligence is a subfolder of the repo; the pipeline expects it as cwd.
    PROJECT = "legallens_repo/legal-doc-intelligence"
else:
    # Option B/C: point PROJECT at the uploaded / mounted project root.
    PROJECT = "legal-doc-intelligence"

assert os.path.isdir(PROJECT), (
    f"Project folder not found: {PROJECT}. "
    "Either the clone failed (check REPO_URL/BRANCH, or the repo is private and needs a token) "
    "or you're on Option B/C and haven't uploaded/mounted the project yet."
)
%cd {PROJECT}
!ls

## 2. Install dependencies

Colab already ships a CUDA build of PyTorch; this installs the rest from `requirements.txt` plus the spaCy and NLTK data the pipeline needs.

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
print("deps installed")

## 3. (Optional) Build the training dataset

Only needed if you want to **train** on Colab and don't already have `artifacts/training_data/full`. A quick 100-doc build is enough to smoke-test the training path; `--full` is the real build (slow). Skip this cell if you only want to run evaluation.

In [ ]:
# Quick build (100 docs/split). For the real run use: --full --output_dir artifacts/training_data/full
!python scripts/build_dataset.py --sample 100 --output_dir artifacts/training_data/sample100 --overwrite

## 4. (Optional) Fine-tune BART on the GPU

Produces `models/legallens-bart-base-e3`. Point `--data` at the build you made above (`artifacts/training_data/full` for the real model, `artifacts/training_data/sample100` for a smoke test). Skip if you uploaded a checkpoint.

In [ ]:
!python -m src.model.train \
    --data artifacts/training_data/full \
    --output_dir models/legallens-bart-base-e3 \
    --epochs 3

## 5. Run the improved evaluation (A–D)

Scores the full pipeline against the published IN-ABS reference summaries with ROUGE, BERTScore, and the new **outcome** metric. If a fine-tuned checkpoint is present it is compared against the `facebook/bart-base` zero-shot baseline; otherwise the baseline runs alone.

In [ ]:
import os

FINETUNED = "models/legallens-bart-base-e3"
SAMPLE = 100            # first N test documents; set to None for the whole split
OUTPUT = "artifacts/eval/colab_test.json"

models = ["facebook/bart-base"]
if os.path.exists(os.path.join(FINETUNED, "model.safetensors")):
    models.append(FINETUNED)
    print("Comparing zero-shot vs fine-tuned:", models)
else:
    print("No fine-tuned checkpoint found — running zero-shot baseline only.")

sample_arg = f"--sample {SAMPLE}" if SAMPLE else ""
!python -m src.eval.evaluate --models {" ".join(models)} --split test {sample_arg} --bertscore --output {OUTPUT}

## 6. Show the results (including outcome preservation)

In [ ]:
import json

report = json.load(open(OUTPUT, encoding="utf-8"))
print(f"Split: {report['split']}  |  documents: {report['documents']}\n")

metrics = ["rouge1", "rouge2", "rougeL", "bertscore_f1", "outcome_present", "outcome_recall"]
header = f"{'model':<34}{'mode':<8}" + "".join(f"{m:>16}" for m in metrics)
print(header)
print("-" * len(header))
for r in report["results"]:
    for mode, s in r["scores"].items():
        row = f"{r['model']:<34}{mode:<8}"
        for m in metrics:
            v = s.get(m)
            row += f"{(v if v is not None else '-'):>16}"
        print(row)

print("\noutcome_present = % of summaries that state the court's order;")
print("outcome_recall  = token recall of the reference's disposition. Both are the")
print("metrics that catch a dropped final order — ROUGE/BERTScore barely move on it.")

## 7. (Optional) Save results / checkpoint back to Drive

```python
from google.colab import drive; drive.mount('/content/drive')
!cp artifacts/eval/colab_test.json /content/drive/MyDrive/
# and, to keep a trained checkpoint:
!cp -r models/legallens-bart-base-e3 /content/drive/MyDrive/
```